# Week 12 Demo: Service Request Escalation
Predict whether a request will require specialist escalation using information available when it arrives. A label of 1 means escalation occurred. This is a fictional dataset.

Read DATA_GUIDE.md first. Keep the raw data unchanged. Use the course .venv. Run cells in order and keep outputs. The supplied estimator and settings are fixed.

## 1. Load and investigate
Check feature roles, missing values, blank text, and the two excluded fields.

In [ ]:
import sys
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
print('Python:', sys.version.split()[0], 'scikit-learn:', sklearn.__version__)
pd.set_option('display.max_columns', None)

In [ ]:
TARGET = 'escalated'
raw_df = pd.read_csv('data/service_requests.csv')
display(raw_df.head())
print(raw_df.shape)

In [ ]:
raw_before = raw_df.copy(deep=True)
working_df = raw_df.copy()
numeric_source = ['requested_units', 'available_units', 'asset_age_years']
invalid = (working_df[numeric_source].lt(0).any(axis=1)
           | ~working_df[TARGET].isin([0, 1]))
rejected_ids = working_df.loc[invalid, 'record_id'].tolist()
working_df = working_df.loc[~invalid].copy()
print('Rejected IDs:', rejected_ids)
print('Retained rows:', len(working_df))
assert len(raw_df) == len(working_df) + len(rejected_ids)
assert working_df['record_id'].is_unique
pd.testing.assert_frame_equal(raw_df, raw_before)
display(working_df.isna().sum().to_frame('missing_values'))
print('Missing or blank summaries:', working_df['summary'].fillna('').str.strip().eq('').sum())

## 2. Separate training and test data
`record_id` identifies records. `actual_work_hours` is known after the work finishes. Exclude both from X. Missing optional values remain valid. The provided rules are fixed and use no learned statistics.

In [ ]:
FEATURES = ['requested_units', 'available_units', 'asset_age_years', 'priority', 'channel', 'summary']
train_df, test_df = train_test_split(working_df, test_size=0.20, random_state=42, stratify=working_df[TARGET])
X_train, X_test = train_df[FEATURES], test_df[FEATURES]
y_train, y_test = train_df[TARGET], test_df[TARGET]
assert set(train_df['record_id']).isdisjoint(test_df['record_id'])
print('Training rows:', len(X_train), 'Test rows:', len(X_test))
display(pd.DataFrame({'train_share':y_train.value_counts(normalize=True), 'test_share':y_test.value_counts(normalize=True)}))

## 3. Define reusable row preparation
Calculate unit_shortage = max(requested_units - available_units, 0). If either input is missing, keep the result missing until the numeric imputer fills it. The same function runs during training and prediction. Text preparation keeps the input one-dimensional and turns missing/blank text into empty strings.

In [ ]:
def add_shortage(frame):
    # Stateless: each row uses only values supplied in that row.
    result = frame.copy()
    result['unit_shortage'] = (result['requested_units'] - result['available_units']).clip(lower=0)
    return result

def prepare_text(series):
    # ColumnTransformer supplies a one-dimensional Series for the scalar selector.
    return series.fillna('').astype(str).str.strip()

## 4. Route each feature group
Numeric: median imputation then standardization. Nominal channel: constant imputation then one-hot encoding. Ordinal priority: most-frequent imputation then the specified order. Text: empty-string preparation then TF-IDF. Use `'summary'`, a scalar selector, for the text branch. The other branches use lists of column names.

In [ ]:
numeric_columns = ['requested_units', 'available_units', 'asset_age_years', 'unit_shortage']
numeric = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])
nominal = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encode', OneHotEncoder(handle_unknown='ignore'))
])
ordinal = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OrdinalEncoder(categories=[['low', 'normal', 'high']],
                              handle_unknown='use_encoded_value', unknown_value=-1))
])
text = Pipeline([
    ('prepare', FunctionTransformer(prepare_text, validate=False, feature_names_out='one-to-one')),
    ('tfidf', TfidfVectorizer(max_features=100, ngram_range=(1, 1), stop_words=None))
])
preprocess = ColumnTransformer([
    ('numeric', numeric, numeric_columns),
    ('nominal', nominal, ['channel']),
    ('ordinal', ordinal, ['priority']),
    ('text', text, 'summary')
], remainder='drop', sparse_threshold=1.0)
model = Pipeline([
    ('engineer', FunctionTransformer(add_shortage, validate=False)),
    ('prepare', preprocess),
    ('classifier', LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', random_state=42))
])
model.fit(X_train, y_train)
print('Pipeline fitted on training rows only.')

## 5. Inspect the combined features
Inspect feature names, row count, width, and finite values. Sparse output saves space when many text/category entries are zero. Avoid converting the entire matrix to a dense array.

In [ ]:
engineered = model.named_steps['engineer'].transform(X_test)
prepared = model.named_steps['prepare'].transform(engineered)
names = model.named_steps['prepare'].get_feature_names_out()
values = prepared.data if hasattr(prepared, 'toarray') else prepared
assert np.isfinite(values).all()
assert prepared.shape == (len(X_test), len(names))
print('Prepared test shape:', prepared.shape, 'Output type:', type(prepared).__name__)
display(pd.DataFrame({'feature_name':names}).head(16))
small = prepared[:3].toarray() if hasattr(prepared, 'toarray') else prepared[:3]
display(pd.DataFrame(small, columns=names).iloc[:, :10])
print('Only a tiny slice is converted to dense form for inspection.')

## 6. Evaluate on held-out rows
Compare a training-majority baseline with the mixed pipeline using the same test rows. Performance criteria: model accuracy must exceed baseline accuracy and positive-class recall must be at least 0.60. These are practice checks, not evidence of operational readiness. Report results honestly even when a criterion fails. Do not tune using test results.

In [ ]:
pred = model.predict(X_test)
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
base_pred = baseline.predict(X_test)

def metrics(actual, predicted) -> dict[str, float]:
    return {
        'accuracy': float(accuracy_score(actual, predicted)),
        'precision (1)': float(precision_score(actual, predicted, zero_division=0)),
        'recall (1)': float(recall_score(actual, predicted, zero_division=0)),
        'F1 (1)': float(f1_score(actual, predicted, zero_division=0)),
    }

# Keep numeric scores for calculations; use the DataFrame to display the comparison.
model_scores = metrics(y_test, pred)
baseline_scores = metrics(y_test, base_pred)
results = pd.DataFrame({'model': model_scores, 'majority_baseline': baseline_scores})
display(results)

accuracy_above_baseline = model_scores['accuracy'] > baseline_scores['accuracy']
recall_meets_target = model_scores['recall (1)'] >= 0.60
meets_performance_criteria = accuracy_above_baseline and recall_meets_target
print('Meets performance criteria (higher accuracy than baseline and recall >= 0.60):', meets_performance_criteria)
cm = confusion_matrix(y_test,pred,labels=[0,1])
display(pd.DataFrame(cm,index=['actual 0','actual 1'],columns=['predicted 0','predicted 1']))
review = test_df[['record_id','summary',TARGET]].copy()
review['predicted'] = pred
mistakes = review.loc[review[TARGET] != review['predicted']]
print('False positives:', cm[0,1], 'False negatives:', cm[1,0])
display(mistakes.head(4))
print('zero_division=0 reports undefined precision/recall as zero. Inspect the confusion matrix too.')

## 7. Verify new and altered observations
Test unseen channel, optional missing data, whitespace-only and unknown-only text. Check the expected derived values and reuse the fitted vocabulary and medians. Also trigger a missing-column error, explain it, and restore the schema. A prediction completing successfully does not show that it is accurate.

In [ ]:
new_cases = pd.DataFrame([
    {'requested_units':8.,'available_units':3.,'asset_age_years':9.,'priority':'high','channel':'portal','summary':'Intermittent failure during operation'},
    {'requested_units':np.nan,'available_units':2.,'asset_age_years':np.nan,'priority':np.nan,'channel':'mobile_app','summary':np.nan},
    {'requested_units':2.,'available_units':6.,'asset_age_years':1.,'priority':'normal','channel':'email','summary':'   '},
    {'requested_units':3.,'available_units':3.,'asset_age_years':4.,'priority':'low','channel':'phone','summary':'zzzxxyyqq'}
])
assert 'mobile_app' not in set(X_train['channel'].dropna())
assert 'zzzxxyyqq' not in model.named_steps['prepare'].named_transformers_['text'].named_steps['tfidf'].vocabulary_
engineered_cases = model.named_steps['engineer'].transform(new_cases)
assert engineered_cases['unit_shortage'].iloc[0] == 5
assert pd.isna(engineered_cases['unit_shortage'].iloc[1])
assert engineered_cases['unit_shortage'].iloc[2] == 0
fitted = model.named_steps['prepare']
vocab_before = fitted.named_transformers_['text'].named_steps['tfidf'].get_feature_names_out().copy()
medians_before = fitted.named_transformers_['numeric'].named_steps['impute'].statistics_.copy()
new_prepared = fitted.transform(engineered_cases)
assert new_prepared.shape[1] == prepared.shape[1]
new_values = new_prepared.data if hasattr(new_prepared,'toarray') else new_prepared
assert np.isfinite(new_values).all()
text_cases = fitted.named_transformers_['text'].transform(new_cases['summary'])
assert text_cases[1:].nnz == 0  # Missing, blank, and unknown-only documents have no recognized tokens.
# Make the prediction array explicit for the display table and shape checks.
new_pred = np.asarray(model.predict(new_cases))
assert new_pred.ndim == 1
assert len(new_pred) == len(new_cases)
assert np.array_equal(vocab_before, fitted.named_transformers_['text'].named_steps['tfidf'].get_feature_names_out())
assert np.array_equal(medians_before, fitted.named_transformers_['numeric'].named_steps['impute'].statistics_)
display(new_cases.assign(predicted=new_pred))
print('PASS: expected shortage values, stable width, finite values, and unchanged fitted state.')
print('PASS: missing/blank/unknown text and unseen nominal category produce predictions.')
# Demonstrate an input-schema failure and correct it without refitting.
try:
    model.predict(new_cases.drop(columns=['channel']))
except (ValueError, KeyError) as error:
    print('Missing-column check:', type(error).__name__, str(error)[:160])
else:
    raise AssertionError('Expected a missing-column error.')
restored = new_cases.copy()  # Restore the documented schema from the original case data.
assert len(model.predict(restored)) == len(restored)
print('PASS: restored schema predicts without refitting.')

## 8. Communicate findings
The pipeline consistently prepared mixed feature types and handled the tested new inputs. Model accuracy exceeded the baseline, but recall fell below the performance goal, so the results do not support relying on it for escalation planning.